# Binomial trees in SigAlg II: contingent claims

This is the second of three notebooks on risk-neutral valuation in SigAlg with binomial trees. It assumes the binomial model from the first notebook, which is rebuilt below so that this notebook runs on its own.

We define contingent claims and construct path-independent European and American options, the path-dependent Asian option, and a custom-built exotic claim. Pricing is the subject of the third notebook.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmyers7/sigalg/blob/main/docs/notebooks/binomial-pricing-model-2.ipynb)

In [6]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX rendering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

## Contingent claims

### The definition of a contingent claim

Let $S_t$ be the price process of an asset, indexed $t\in \{0,1,\ldots,T\}$ out to some time horizon $T$. A *contingent claim* (or simply a *claim*) is a non-negative stochastic process $\Phi_t$ *adapted* to the filtration $\mathcal{F}_t$ introduced in the first notebook. This just means that each $\Phi_t$ is $\mathcal{F}_t$-measurable (for all $t$), which is equivalent to requiring that it is a function of all prices up to (and including) $S_t$:
$$
\Phi_t = \Phi_t(S_0,S_1,\ldots,S_t).
$$
The claim is called *path-independent* if each $\Phi_t$ is $S_t$-measurable (for all $t$), which means that it is a function of the current price alone:
$$
\Phi_t = \Phi_t(S_t).
$$
The time horizon $T$ is called the *maturity time* of the claim. The claim is said to be a *non-early-exercise* claim if $\Phi_t = 0$ for all non-maturity times $t<T$; otherwise, the claim is said to *allow early exercise*.

### European options

One of the simplest examples of a non-early-exercise claim is a *European call option* given by
$$
\Phi_T = \max(S_T-K,0),
$$
where $K$ is a positive real number called the *strike price* of the claim. We may implement this claim in SigAlg by hand using the `from_random_variables` constructor of the `Claim` class. We use a binomial model as the underlying price process $S_t$.

In [7]:
import sigalg as sa

S_0, u, d, p_u, R, T = 100, 1.1, 0.9, 0.7, 1.05, 3

S = sa.BinomialPricingModel.generate(
    initial_price=S_0,
    up_factor=u,
    down_factor=d,
    up_prob=p_u,
    risk_free_gross_return=R,
    mode="enum",
    length=T,
    name="S",
)

K = 100
Phi_T = (S[T] - K).max(0)

euro_call = sa.Claim.from_random_variables(
    rvs=[Phi_T], times=[T], model=S, name="euro_call"
)

print("The asset pricing model:")
print(S)

print("\nThe claim payoff at maturity:")
print(Phi_T)

print("\nThe European call option:")
print(euro_call)

The asset pricing model:
Binomial price process 'S':
t          0      1      2      3
omega                            
0      100.0  110.0  121.0  133.1
1      100.0  110.0  121.0  108.9
2      100.0  110.0   99.0  108.9
3      100.0  110.0   99.0   89.1
4      100.0   90.0   99.0  108.9
5      100.0   90.0   99.0   89.1
6      100.0   90.0   81.0   89.1
7      100.0   90.0   81.0   72.9

The claim payoff at maturity:
Random variable 'max((S_3 - 100), 0)':
       max((S_3 - 100), 0)
omega                     
0                     33.1
1                      8.9
2                      8.9
3                      0.0
4                      8.9
5                      0.0
6                      0.0
7                      0.0

The European call option:
Claim 'euro_call':
t        0    1    2     3
omega                     
0      0.0  0.0  0.0  33.1
1      0.0  0.0  0.0   8.9
2      0.0  0.0  0.0   8.9
3      0.0  0.0  0.0   0.0
4      0.0  0.0  0.0   8.9
5      0.0  0.0  0.0   0.0
6    

A *European put option* is a non-early-exercise claim with
$$
\Phi_T = \max(K-S_T,0).
$$
It may be implemented in SigAlg by hand just as easily as the call option above; however, SigAlg also includes convenient factory methods for typical contingent claims. The next code block uses one of these methods to construct a European put option in one line.

In [8]:
euro_put = sa.Claim.european_option(
    strike=K, model=S, option_type="put", name="euro_put"
)

print("The asset pricing model:")
print(S)

print("\nThe European put option:")
print(euro_put)

The asset pricing model:
Binomial price process 'S':
t          0      1      2      3
omega                            
0      100.0  110.0  121.0  133.1
1      100.0  110.0  121.0  108.9
2      100.0  110.0   99.0  108.9
3      100.0  110.0   99.0   89.1
4      100.0   90.0   99.0  108.9
5      100.0   90.0   99.0   89.1
6      100.0   90.0   81.0   89.1
7      100.0   90.0   81.0   72.9

The European put option:
European option 'euro_put':
t        0    1    2     3
omega                     
0      0.0  0.0  0.0   0.0
1      0.0  0.0  0.0   0.0
2      0.0  0.0  0.0   0.0
3      0.0  0.0  0.0  10.9
4      0.0  0.0  0.0   0.0
5      0.0  0.0  0.0  10.9
6      0.0  0.0  0.0  10.9
7      0.0  0.0  0.0  27.1


### American options

American options are the early-exercise analogues of European options. In particular, an *American call option* with strike $K$ is a claim with

$$
\Phi_t = \max(S_t-K,0)
$$

for all $t$, while an *American put option* with strike $K$ is given by

$$
\Phi_t = \max(K-S_t, 0)
$$

for all $t$. In the first of the next two code blocks, we build an American call option by hand to again illustrate the naturality of the SigAlg API, and then in the second we use a factory method to build an American put option in one line.

In [9]:
Phi = (S - K).max(0)

american_call = sa.Claim.from_stochastic_process(
    process=Phi, model=S, name="american_call"
)

print("The asset pricing model:")
print(S)

print("\nThe American call option:")
print(american_call)

The asset pricing model:
Binomial price process 'S':
t          0      1      2      3
omega                            
0      100.0  110.0  121.0  133.1
1      100.0  110.0  121.0  108.9
2      100.0  110.0   99.0  108.9
3      100.0  110.0   99.0   89.1
4      100.0   90.0   99.0  108.9
5      100.0   90.0   99.0   89.1
6      100.0   90.0   81.0   89.1
7      100.0   90.0   81.0   72.9

The American call option:
Claim 'american_call':
t        0     1     2     3
omega                       
0      0.0  10.0  21.0  33.1
1      0.0  10.0  21.0   8.9
2      0.0  10.0   0.0   8.9
3      0.0  10.0   0.0   0.0
4      0.0   0.0   0.0   8.9
5      0.0   0.0   0.0   0.0
6      0.0   0.0   0.0   0.0
7      0.0   0.0   0.0   0.0


In [10]:
american_put = sa.Claim.american_option(
    strike=K, model=S, option_type="put", name="american_put"
)

print("The asset pricing model:")
print(S)

print("\nThe American put option:")
print(american_put)

The asset pricing model:
Binomial price process 'S':
t          0      1      2      3
omega                            
0      100.0  110.0  121.0  133.1
1      100.0  110.0  121.0  108.9
2      100.0  110.0   99.0  108.9
3      100.0  110.0   99.0   89.1
4      100.0   90.0   99.0  108.9
5      100.0   90.0   99.0   89.1
6      100.0   90.0   81.0   89.1
7      100.0   90.0   81.0   72.9

The American put option:
American option 'american_put':
t        0     1     2     3
omega                       
0      0.0   0.0   0.0   0.0
1      0.0   0.0   0.0   0.0
2      0.0   0.0   1.0   0.0
3      0.0   0.0   1.0  10.9
4      0.0  10.0   1.0   0.0
5      0.0  10.0   1.0  10.9
6      0.0  10.0  19.0  10.9
7      0.0  10.0  19.0  27.1


### Asian options

European and American options are path-independent claims. An example of a path-dependent claim is an *Asian call option* with strike $K$, which is given by
$$
\Phi_t = \max(\overline{S}_t- K,0)
$$
for all $t$, where
$$
\overline{S}_t = \frac{1}{1+t} \sum_{k=0}^t S_k
$$
for each $t$. Similarly, an *Asian put option* with strike $K$ is given by
$$
\Phi_t = \max(K-\overline{S}_t,0).
$$
The next pair of code cells follows the pattern above: First constructing an Asian call option by hand, then using a factory method to construct an Asian put option.

In [11]:
Phi = (S.cumavg() - K).max(0)

asian_call = sa.Claim.from_stochastic_process(process=Phi, model=S, name="asian_call")

print("The asset pricing model:")
print(S)

print("\nThe Asian call option:")
print(asian_call)

The asset pricing model:
Binomial price process 'S':
t          0      1      2      3
omega                            
0      100.0  110.0  121.0  133.1
1      100.0  110.0  121.0  108.9
2      100.0  110.0   99.0  108.9
3      100.0  110.0   99.0   89.1
4      100.0   90.0   99.0  108.9
5      100.0   90.0   99.0   89.1
6      100.0   90.0   81.0   89.1
7      100.0   90.0   81.0   72.9

The Asian call option:
Claim 'asian_call':
t        0    1          2       3
omega                             
0      0.0  5.0  10.333333  16.025
1      0.0  5.0  10.333333   9.975
2      0.0  5.0   3.000000   4.475
3      0.0  5.0   3.000000   0.000
4      0.0  0.0   0.000000   0.000
5      0.0  0.0   0.000000   0.000
6      0.0  0.0   0.000000   0.000
7      0.0  0.0   0.000000   0.000


In [12]:
asian_put = sa.Claim.asian_option(
    strike=K, model=S, option_type="put", name="asian_put"
)

print("The asset pricing model:")
print(S)

print("\nThe Asian put option:")
print(asian_put)

The asset pricing model:
Binomial price process 'S':
t          0      1      2      3
omega                            
0      100.0  110.0  121.0  133.1
1      100.0  110.0  121.0  108.9
2      100.0  110.0   99.0  108.9
3      100.0  110.0   99.0   89.1
4      100.0   90.0   99.0  108.9
5      100.0   90.0   99.0   89.1
6      100.0   90.0   81.0   89.1
7      100.0   90.0   81.0   72.9

The Asian put option:
Asian option 'asian_put':
t        0    1    2       3
omega                       
0      0.0  0.0  0.0   0.000
1      0.0  0.0  0.0   0.000
2      0.0  0.0  0.0   0.000
3      0.0  0.0  0.0   0.475
4      0.0  0.0  0.0   0.525
5      0.0  0.0  0.0   5.475
6      0.0  0.0  0.0   9.975
7      0.0  0.0  0.0  14.025


### Exotic claims

The SigAlg API allows the user to construct claims with all different types of payoff structures. How about a claim that is a mix of an Asian call and a European put, and only allows exercise at two specified times?

In [13]:
Phi_1 = (K - S[1]).max(0)
Phi_3 = (S.cumavg()[3] - K).max(0)

exotic_claim = sa.Claim.from_random_variables(
    rvs=[Phi_1, Phi_3], times=[1, 3], model=S, name="exotic_claim"
)

print("The asset pricing model:")
print(S)

print("\nThe exotic claim:")
print(exotic_claim)

The asset pricing model:
Binomial price process 'S':
t          0      1      2      3
omega                            
0      100.0  110.0  121.0  133.1
1      100.0  110.0  121.0  108.9
2      100.0  110.0   99.0  108.9
3      100.0  110.0   99.0   89.1
4      100.0   90.0   99.0  108.9
5      100.0   90.0   99.0   89.1
6      100.0   90.0   81.0   89.1
7      100.0   90.0   81.0   72.9

The exotic claim:
Claim 'exotic_claim':
t        0     1    2       3
omega                        
0      0.0   0.0  0.0  16.025
1      0.0   0.0  0.0   9.975
2      0.0   0.0  0.0   4.475
3      0.0   0.0  0.0   0.000
4      0.0  10.0  0.0   0.000
5      0.0  10.0  0.0   0.000
6      0.0  10.0  0.0   0.000
7      0.0  10.0  0.0   0.000
